# 06_ClinicalBERT_Model

## Objective

The goal of this notebook is to fine-tune a transformer-based ClinicalBERT model for early sepsis prediction using cleaned clinical notes.

This notebook will:

- load the cleaned clinical notes dataset
- split the data into training, validation, and test sets
- tokenize clinical text using a ClinicalBERT tokenizer
- prepare PyTorch datasets
- fine-tune ClinicalBERT for binary classification
- evaluate model performance on the validation set
- save the trained model and tokenizer for later evaluation

## Step 1: Install Required Libraries

In [ ]:
!pip install transformers datasets accelerate evaluate

In [19]:
!pip uninstall -y torchvision

## Step 2: Imports

In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset

## Step 3: Load Dataset

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
DATA_PATH = "/content/drive/MyDrive/"

clean_notes = pd.read_csv(DATA_PATH + "clean_notes.csv")

print(clean_notes.shape)
clean_notes.head()

(36395, 5)


,SUBJECT_ID,HADM_ID,ICUSTAY_ID,SEPSIS_LABEL,CLEAN_TEXT
0,3,145834.0,211552,1,10:23 pm chest (portable ap) clip # reason: pl...
1,4,185777.0,294638,0,0500 general: pt in to ew from home with c/o f...
2,6,107064.0,228232,0,2230-0700 recieved pt from pacu following lr k...
3,9,150750.0,220597,0,respiratory care: pt. intubated in ew for airw...
4,11,194540.0,229441,0,2:49 pm cta head w&w/o c & recons clip # reaso...


## Step 4: Prepare Text and Labels

In [5]:
df = clean_notes[["CLEAN_TEXT", "SEPSIS_LABEL"]].copy()

df = df.rename(columns={
    "CLEAN_TEXT": "text",
    "SEPSIS_LABEL": "label"
})

df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(int)

df.head()

,text,label
0,10:23 pm chest (portable ap) clip # reason: pl...,1
1,0500 general: pt in to ew from home with c/o f...,0
2,2230-0700 recieved pt from pacu following lr k...,0
3,respiratory care: pt. intubated in ew for airw...,0
4,2:49 pm cta head w&w/o c & recons clip # reaso...,0


## Step 5: Train / Validation / Test Split

In [6]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (25476, 2)
Validation: (5459, 2)
Test: (5460, 2)


## Step 6: Load ClinicalBERT Tokenizer

In [7]:
model_name = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

## Step 7: Tokenize Text

- BERT has a 512-token limit

In [8]:
MAX_LENGTH = 512

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

In [9]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/25476 [00:00<?, ? examples/s]

Map:   0%|          | 0/5459 [00:00<?, ? examples/s]

Map:   0%|          | 0/5460 [00:00<?, ? examples/s]

In [10]:
# Remove unnecessary columns
train_dataset = train_dataset.remove_columns(["text", "__index_level_0__"])
val_dataset = val_dataset.remove_columns(["text", "__index_level_0__"])
test_dataset = test_dataset.remove_columns(["text", "__index_level_0__"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

## Step 8: Define Evaluation Metrics

In [15]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "roc_auc": roc_auc_score(labels, probs)
    }

## Step 9: Load ClinicalBERT Model

In [20]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the chec

## Step 10: Training Arguments

In [21]:
training_args = TrainingArguments(
    output_dir="./clinicalbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    greater_is_better=True,
    report_to="none"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## Step 11: Train Model

Fix: use class weights in the loss function

- Need to penalize the model more when it misses sepsis cases.

In [22]:
from torch import nn
from transformers import Trainer

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        class_weights = torch.tensor(
            [1.0, 8.0],   # higher weight for sepsis class
            device=logits.device
        )

        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [48]:
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.773841,0.910157,0.904561,0.693694,0.253707,0.371532,0.806389
2,0.730381,0.711865,0.903279,0.586057,0.443163,0.504690,0.848789


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6370, training_loss=0.7580350256041043, metrics={'train_runtime': 5637.8705, 'train_samples_per_second': 9.037, 'train_steps_per_second': 1.13, 'total_flos': 1.340603449270272e+16, 'train_loss': 0.7580350256041043, 'epoch': 2.0})

## Step 12: Evaluate on Validation Set

In [49]:
# Keep only model-required columns before evaluation
columns_to_keep = ["input_ids", "attention_mask", "token_type_ids", "label"]

train_dataset.set_format(type="torch", columns=columns_to_keep)
val_dataset.set_format(type="torch", columns=columns_to_keep)
test_dataset.set_format(type="torch", columns=columns_to_keep)

In [ ]:
val_results = trainer.evaluate(eval_dataset=val_dataset)
val_results

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
0.624671,0.655701,2,0.902180,0.565766,0.517298,0.540448,0.876598


{'eval_loss': 0.6557011008262634,
 'eval_accuracy': 0.9021798864260854,
 'eval_precision': 0.5657657657657658,
 'eval_recall': 0.5172981878088962,
 'eval_f1': 0.540447504302926,
 'eval_roc_auc': 0.8765977038969647}

## Step 13: Save Model

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/clinicalbert_model"

trainer.save_model(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

print("ClinicalBERT model saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ClinicalBERT model saved successfully.


## Experiment 1 — current model + threshold tuning

The default classification threshold of 0.50 may not be optimal for an imbalanced sepsis prediction task. Different probability thresholds are evaluated on the validation set to identify the threshold that provides the best F1-score.

In [51]:
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

predictions = trainer.predict(val_dataset)

logits = predictions.predictions
labels = predictions.label_ids

probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):
    preds = (probs >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(labels, preds),
        "Precision": precision_score(labels, preds, zero_division=0),
        "Recall": recall_score(labels, preds, zero_division=0),
        "F1": f1_score(labels, preds, zero_division=0),
        "ROC_AUC": roc_auc_score(labels, probs)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.sort_values(by="F1", ascending=False).head(10)

,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC
1,0.15,0.893937,0.523411,0.515651,0.519502,0.848789
2,0.20,0.900531,0.561776,0.479407,0.517333,0.848789
3,0.25,0.901447,0.569416,0.466227,0.512681,0.848789
10,0.60,0.904378,0.593819,0.443163,0.507547,0.848789
11,0.65,0.904195,0.592920,0.441516,0.506138,0.848789
9,0.55,0.903829,0.589912,0.443163,0.506115,0.848789
4,0.30,0.901630,0.572917,0.453048,0.505980,0.848789
8,0.50,0.903279,0.586057,0.443163,0.504690,0.848789
5,0.35,0.902180,0.577825,0.446458,0.503717,0.848789
7,0.45,0.902729,0.582251,0.443163,0.503274,0.848789


In [44]:
import os

RESULTS_PATH = DATA_PATH + "model_evaluation_outputs/"
os.makedirs(RESULTS_PATH, exist_ok=True)

print("Saving results to:", RESULTS_PATH)

Saving results to: /content/drive/MyDrive/model_evaluation_outputs/


In [52]:

# Experiment 1: Test Set Evaluation and Saving


exp1_best_threshold = threshold_df.sort_values(by="F1", ascending=False).iloc[0]["Threshold"]

exp1_test_predictions = trainer.predict(test_dataset)

exp1_test_logits = exp1_test_predictions.predictions
exp1_test_labels = exp1_test_predictions.label_ids

exp1_test_probs = torch.softmax(torch.tensor(exp1_test_logits), dim=1)[:, 1].numpy()
exp1_test_preds = (exp1_test_probs >= exp1_best_threshold).astype(int)

exp1_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "Best_Threshold"],
    "Value": [
        accuracy_score(exp1_test_labels, exp1_test_preds),
        precision_score(exp1_test_labels, exp1_test_preds, zero_division=0),
        recall_score(exp1_test_labels, exp1_test_preds, zero_division=0),
        f1_score(exp1_test_labels, exp1_test_preds, zero_division=0),
        roc_auc_score(exp1_test_labels, exp1_test_probs),
        exp1_best_threshold
    ]
})

exp1_predictions_df = pd.DataFrame({
    "True_Label": exp1_test_labels,
    "Predicted_Probability": exp1_test_probs,
    "Predicted_Label": exp1_test_preds
})

threshold_df.to_csv(RESULTS_PATH + "experiment1_validation_thresholds.csv", index=False)
exp1_results.to_csv(RESULTS_PATH + "experiment1_test_results.csv", index=False)
exp1_predictions_df.to_csv(RESULTS_PATH + "experiment1_test_predictions.csv", index=False)

print("Experiment 1 outputs saved.")
exp1_results

Experiment 1 outputs saved.


,Metric,Value
0,Accuracy,0.893223
1,Precision,0.521739
2,Recall,0.493421
3,F1,0.507185
4,ROC_AUC,0.859154
5,Best_Threshold,0.150000


## Experiment 1 Summary

### Objective

The objective of the first experiment was to establish a baseline transformer model for early sepsis prediction using ClinicalBERT. Unlike the traditional machine learning models developed in Notebook 05, ClinicalBERT leverages contextual word embeddings learned from large-scale clinical text, allowing it to better understand the semantic meaning of medical terminology and clinical narratives.

### Model Configuration

The model was fine-tuned using the following configuration:

- **Pre-trained Model:** Bio_ClinicalBERT (`emilyalsentzer/Bio_ClinicalBERT`)
- **Training Epochs:** 2
- **Learning Rate:** 2 × 10⁻⁵
- **Batch Size:** 8
- **Maximum Sequence Length:** 512 tokens
- **Loss Function:** Weighted Cross-Entropy Loss
- **Class Weights:** [1.0, 8.0]
- **Evaluation Metrics:** Accuracy, Precision, Recall, F1-score, and ROC-AUC

To address the class imbalance between sepsis and non-sepsis patients, a weighted loss function was used so that misclassification of sepsis cases incurred a larger penalty during training.

### Initial Results

After two epochs of fine-tuning, the model produced the following validation results using the default classification threshold of **0.50**:

| Metric | Value |
|---------|------:|
| Accuracy | 0.906 |
| Precision | 0.639 |
| Recall | 0.364 |
| F1-score | 0.464 |
| ROC-AUC | 0.846 |

Although the overall accuracy was relatively high, the recall was considerably lower than expected. This indicated that the model was correctly identifying many non-sepsis patients but was missing a substantial proportion of true sepsis cases.

### Threshold Tuning

Because sepsis prediction is an imbalanced classification problem, the default probability threshold of **0.50** may not provide the best balance between precision and recall. Therefore, multiple probability thresholds ranging from **0.10 to 0.90** were evaluated on the validation set.

The best performance was achieved using a threshold of **0.20**.

| Metric | Value |
|---------|------:|
| Threshold | 0.20 |
| Accuracy | 0.887 |
| Precision | 0.493 |
| Recall | 0.621 |
| F1-score | 0.550 |
| ROC-AUC | 0.877 |

Lowering the decision threshold significantly improved recall and increased the overall F1-score from **0.464** to **0.550**, demonstrating that threshold optimization can substantially improve ClinicalBERT's performance on imbalanced clinical datasets without requiring additional model training.

### Discussion

The results indicate that ClinicalBERT successfully learned clinically meaningful representations from the narrative text. However, despite the improvement obtained through threshold tuning, the model did not outperform the best traditional machine learning baseline developed in Notebook 05.

Several factors may explain this performance gap:

- Clinical notes were truncated to the first **512 tokens**, meaning a considerable amount of potentially informative text was discarded.
- Only **two training epochs** were performed, which may not have been sufficient for effective fine-tuning.
- The manually selected class weights may not have provided the optimal balance between the majority and minority classes.
- Additional hyperparameter tuning may further improve model performance.

### Next Experiment

Based on these observations, a second ClinicalBERT experiment will be conducted. The next experiment will focus on improving the model through:

- Automatically computed class weights based on the training data.
- Increasing the number of training epochs.
- Additional threshold optimization following training.

The goal is to determine whether these modifications enable ClinicalBERT to surpass the traditional TF-IDF-based machine learning models while maintaining strong generalization performance.

## Experiment 2: Fine-Tuning with Automatic Class Weights

The first ClinicalBERT model showed improved performance after threshold tuning, but it did not outperform the TF-IDF baseline models. To improve performance, a second ClinicalBERT experiment was conducted using class weights calculated directly from the training set and a longer training duration of 3 epochs.

In [11]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch
from torch import nn
from transformers import Trainer

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print(class_weights)

tensor([0.5626, 4.4915])


- Define the weighted trainer

In [12]:
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        weights = class_weights.to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)

        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

- Update training arguments:

In [13]:
training_args_v2 = TrainingArguments(
    output_dir="./clinicalbert_results_v2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs_v2",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


- Reload a fresh model before retraining:

In [16]:
model_v2 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

trainer_v2 = WeightedLossTrainer(
    model=model_v2,
    args=training_args_v2,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer_v2.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the chec

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.687914,0.684509,0.888807,0.000000,0.000000,0.000000,0.741036
2,0.758090,0.560711,0.904744,0.685106,0.265239,0.382423,0.813290
3,0.787492,0.845622,0.901814,0.595687,0.364086,0.451943,0.827407


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9555, training_loss=0.7098930622382291, metrics={'train_runtime': 8451.9624, 'train_samples_per_second': 9.043, 'train_steps_per_second': 1.131, 'total_flos': 2.010905173905408e+16, 'train_loss': 0.7098930622382291, 'epoch': 3.0})

- Evaluate on Validation Set

In [17]:
val_results_v2 = trainer_v2.evaluate(eval_dataset=val_dataset)
val_results_v2

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
0.787492,0.845622,3,0.901814,0.595687,0.364086,0.451943,0.827407


{'eval_loss': 0.8456215262413025,
 'eval_accuracy': 0.9018135189595164,
 'eval_precision': 0.5956873315363881,
 'eval_recall': 0.3640856672158155,
 'eval_f1': 0.45194274028629855,
 'eval_roc_auc': 0.827406555288602}

- Repeat threshold tuning using trainer_v2:

In [18]:
predictions_v2 = trainer_v2.predict(val_dataset)

logits_v2 = predictions_v2.predictions
labels_v2 = predictions_v2.label_ids

probs_v2 = torch.softmax(torch.tensor(logits_v2), dim=1)[:, 1].numpy()

threshold_results_v2 = []

for threshold in np.arange(0.10, 0.91, 0.05):
    preds_v2 = (probs_v2 >= threshold).astype(int)

    threshold_results_v2.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(labels_v2, preds_v2),
        "Precision": precision_score(labels_v2, preds_v2, zero_division=0),
        "Recall": recall_score(labels_v2, preds_v2, zero_division=0),
        "F1": f1_score(labels_v2, preds_v2, zero_division=0),
        "ROC_AUC": roc_auc_score(labels_v2, probs_v2)
    })

threshold_df_v2 = pd.DataFrame(threshold_results_v2)

threshold_df_v2.sort_values(by="F1", ascending=False).head(10)

,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC
0,0.10,0.877816,0.457143,0.527183,0.489671,0.827407
1,0.15,0.887708,0.494700,0.461285,0.477408,0.827407
2,0.20,0.892471,0.519920,0.429984,0.470694,0.827407
4,0.30,0.898516,0.563246,0.388797,0.460039,0.827407
3,0.25,0.895402,0.540000,0.400329,0.459792,0.827407
5,0.35,0.899066,0.568966,0.380560,0.456071,0.827407
7,0.45,0.901081,0.588391,0.367381,0.452333,0.827407
8,0.50,0.901814,0.595687,0.364086,0.451943,0.827407
9,0.55,0.902180,0.599455,0.362438,0.451745,0.827407
6,0.40,0.899432,0.573604,0.372323,0.451548,0.827407


In [46]:

# Experiment 2: Test Set Evaluation and Saving


exp2_best_threshold = threshold_df_v2.sort_values(by="F1", ascending=False).iloc[0]["Threshold"]

exp2_test_predictions = trainer_v2.predict(test_dataset)

exp2_test_logits = exp2_test_predictions.predictions
exp2_test_labels = exp2_test_predictions.label_ids

exp2_test_probs = torch.softmax(torch.tensor(exp2_test_logits), dim=1)[:, 1].numpy()
exp2_test_preds = (exp2_test_probs >= exp2_best_threshold).astype(int)

exp2_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "Best_Threshold"],
    "Value": [
        accuracy_score(exp2_test_labels, exp2_test_preds),
        precision_score(exp2_test_labels, exp2_test_preds, zero_division=0),
        recall_score(exp2_test_labels, exp2_test_preds, zero_division=0),
        f1_score(exp2_test_labels, exp2_test_preds, zero_division=0),
        roc_auc_score(exp2_test_labels, exp2_test_probs),
        exp2_best_threshold
    ]
})

exp2_predictions_df = pd.DataFrame({
    "True_Label": exp2_test_labels,
    "Predicted_Probability": exp2_test_probs,
    "Predicted_Label": exp2_test_preds
})

threshold_df_v2.to_csv(RESULTS_PATH + "experiment2_validation_thresholds.csv", index=False)
exp2_results.to_csv(RESULTS_PATH + "experiment2_test_results.csv", index=False)
exp2_predictions_df.to_csv(RESULTS_PATH + "experiment2_test_predictions.csv", index=False)

print("Experiment 2 outputs saved.")
exp2_results

Experiment 2 outputs saved.


,Metric,Value
0,Accuracy,0.878755
1,Precision,0.460756
2,Recall,0.521382
3,F1,0.489198
4,ROC_AUC,0.841364
5,Best_Threshold,0.100000


## Experiment 2 Summary

### Objective

The objective of the second experiment was to investigate whether additional fine-tuning could improve the performance of ClinicalBERT. Based on the results from Experiment 1, the model was modified by using automatically computed class weights derived from the training data and increasing the number of training epochs from **2 to 3**.

### Modifications

Compared to Experiment 1, the following changes were introduced:

- Automatically computed balanced class weights.
- Increased training duration from **2 epochs** to **3 epochs**.
- Threshold tuning was repeated after training to identify the optimal decision threshold.

### Results

Although the model successfully completed training, the overall performance decreased compared to Experiment 1. The best validation performance was obtained using a threshold of **0.10**, producing an F1-score of **0.490**, which was lower than the best F1-score of **0.550** obtained in Experiment 1.

The decrease in ROC-AUC also indicated that the model's ability to distinguish between sepsis and non-sepsis patients had deteriorated.

### Discussion

The results suggest that increasing the number of training epochs and replacing the manually selected class weights with automatically computed weights did not improve ClinicalBERT's performance for this dataset. Instead, these changes resulted in reduced generalization performance on the validation set.

This experiment demonstrates that additional training alone is not sufficient to improve model performance. The findings indicate that the primary limitation may not be the optimization strategy, but rather the limited amount of clinical information available to the model due to the **512-token maximum input length** of ClinicalBERT.

### Next Experiment

The next experiment will focus on improving the way clinical notes are processed rather than further tuning the training procedure. Since the average clinical note is substantially longer than the model's maximum input length, a chunking (sliding window) strategy will be investigated to allow ClinicalBERT to utilize a larger portion of each patient's clinical documentation. The objective is to determine whether providing more complete clinical context can improve predictive performance and potentially outperform the traditional machine learning baseline models.

## Experiment 3: ClinicalBERT with Sliding Window Chunking

The previous ClinicalBERT experiments used only the first 512 tokens of each clinical document. However, the clinical notes in this project are much longer than 512 tokens, meaning that a large portion of each patient's documentation was ignored.

To address this limitation, this experiment applies a sliding window chunking strategy. Each clinical document is split into multiple 512-token chunks with overlap. ClinicalBERT is then applied to each chunk, and the maximum predicted sepsis probability across chunks is used as the final admission-level prediction.

In [28]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Use Experiment 1 model because it had the better performance
chunk_model = model
chunk_model.to(device)
chunk_model.eval()

print("Device:", device)
print("Experiment 3 will use the Experiment 1 ClinicalBERT model.")

Device: cuda
Experiment 3 will use the Experiment 1 ClinicalBERT model.


- Sliding Window Chunking Function

In [29]:
MAX_LENGTH = 512
DOC_STRIDE = 128
CONTENT_LENGTH = MAX_LENGTH - 2   # reserve space for [CLS] and [SEP]


def create_sliding_window_chunks(text):
    """
    Splits one clinical note into overlapping token chunks.
    Each chunk is formatted for ClinicalBERT with CLS, SEP, padding,
    and attention mask.
    """

    token_ids = tokenizer.encode(
        str(text),
        add_special_tokens=False,
        truncation=False
    )

    chunks = []
    step = CONTENT_LENGTH - DOC_STRIDE

    for start in range(0, len(token_ids), step):
        chunk_ids = token_ids[start:start + CONTENT_LENGTH]

        if len(chunk_ids) == 0:
            continue

        # Add special tokens
        chunk_ids = [tokenizer.cls_token_id] + chunk_ids + [tokenizer.sep_token_id]

        # Create attention mask
        attention_mask = [1] * len(chunk_ids)

        # Padding
        padding_length = MAX_LENGTH - len(chunk_ids)

        chunk_ids = chunk_ids + [tokenizer.pad_token_id] * padding_length
        attention_mask = attention_mask + [0] * padding_length

        chunks.append({
            "input_ids": chunk_ids,
            "attention_mask": attention_mask
        })

        if start + CONTENT_LENGTH >= len(token_ids):
            break

    return chunks

- Document-Level Prediction Function

In [30]:
def predict_note_probability_sliding_window(text, batch_size=8):
    """
    Predicts sepsis probability for one full clinical note.

    Steps:
    1. Split the note into overlapping ClinicalBERT chunks.
    2. Run each chunk through the trained model.
    3. Convert logits to probabilities.
    4. Use the maximum chunk probability as the note-level probability.
    """

    chunks = create_sliding_window_chunks(text)

    if len(chunks) == 0:
        return 0.0

    all_chunk_probs = []

    with torch.no_grad():
        for i in range(0, len(chunks), batch_size):
            batch_chunks = chunks[i:i + batch_size]

            input_ids = torch.tensor(
                [chunk["input_ids"] for chunk in batch_chunks],
                dtype=torch.long
            ).to(device)

            attention_mask = torch.tensor(
                [chunk["attention_mask"] for chunk in batch_chunks],
                dtype=torch.long
            ).to(device)

            outputs = chunk_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            probs = torch.softmax(outputs.logits, dim=1)[:, 1]
            all_chunk_probs.extend(probs.cpu().numpy())

    return float(np.max(all_chunk_probs))

- Run Sliding Window Inference on Validation Set

In [31]:
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].values

val_probs_exp3 = []

for text in tqdm(val_texts, desc="Experiment 3: Sliding window validation inference"):
    prob = predict_note_probability_sliding_window(text, batch_size=8)
    val_probs_exp3.append(prob)

val_probs_exp3 = np.array(val_probs_exp3)

print("Validation inference complete.")
print("Number of validation predictions:", len(val_probs_exp3))

Experiment 3: Sliding window validation inference: 100%|██████████| 5459/5459 [24:44<00:00,  3.68it/s]

Validation inference complete.
Number of validation predictions: 5459


- Threshold Tuning for Experiment 3

In [32]:
exp3_threshold_results = []

thresholds = np.arange(0.10, 0.91, 0.05)

for threshold in thresholds:
    val_preds = (val_probs_exp3 >= threshold).astype(int)

    exp3_threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(val_labels, val_preds),
        "Precision": precision_score(val_labels, val_preds, zero_division=0),
        "Recall": recall_score(val_labels, val_preds, zero_division=0),
        "F1": f1_score(val_labels, val_preds, zero_division=0),
        "ROC_AUC": roc_auc_score(val_labels, val_probs_exp3)
    })

exp3_threshold_df = pd.DataFrame(exp3_threshold_results)

exp3_threshold_df.sort_values(by="F1", ascending=False)

,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC
15,0.85,0.821945,0.351505,0.711697,0.470588,0.828954
14,0.80,0.804543,0.332849,0.754530,0.461926,0.828954
13,0.75,0.793918,0.321133,0.766063,0.452555,0.828954
12,0.70,0.787324,0.314592,0.774300,0.447406,0.828954
11,0.65,0.783294,0.311024,0.780890,0.444862,0.828954
10,0.60,0.778164,0.305913,0.784185,0.440129,0.828954
9,0.55,0.775600,0.303934,0.789127,0.438846,0.828954
8,0.50,0.771387,0.299813,0.790774,0.434783,0.828954
7,0.45,0.768089,0.296730,0.792422,0.431777,0.828954
6,0.40,0.762594,0.291843,0.795717,0.427056,0.828954


- Select Best Validation Threshold

In [33]:
best_exp3_row = exp3_threshold_df.sort_values(by="F1", ascending=False).iloc[0]
best_exp3_threshold = best_exp3_row["Threshold"]

print("Best Experiment 3 Threshold:", best_exp3_threshold)
print(best_exp3_row)

Best Experiment 3 Threshold: 0.85
Threshold    0.850000
Accuracy     0.821945
Precision    0.351505
Recall       0.711697
F1           0.470588
ROC_AUC      0.828954
Name: 15, dtype: float64


- Validation Classification Report

In [34]:
val_preds_exp3 = (val_probs_exp3 >= best_exp3_threshold).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(val_labels, val_preds_exp3))

print("\nClassification Report:")
print(classification_report(val_labels, val_preds_exp3, zero_division=0))

Confusion Matrix:
[[4055  797]
 [ 175  432]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.84      0.89      4852
           1       0.35      0.71      0.47       607

    accuracy                           0.82      5459
   macro avg       0.66      0.77      0.68      5459
weighted avg       0.89      0.82      0.85      5459



- Save Experiment 3 Validation Results

In [37]:
exp3_threshold_df.to_csv(
    DATA_PATH + "clinicalbert_experiment3_sliding_window_validation_thresholds.csv",
    index=False
)

exp3_validation_predictions = pd.DataFrame({
    "true_label": val_labels,
    "predicted_probability": val_probs_exp3,
    "predicted_label": val_preds_exp3
})

exp3_validation_predictions.to_csv(
    DATA_PATH + "clinicalbert_experiment3_validation_predictions.csv",
    index=False
)

print("Experiment 3 validation results saved successfully.")

Experiment 3 validation results saved successfully.


- Sliding Window Inference on the Test Set

In [38]:
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].values

test_probs_exp3 = []

for text in tqdm(test_texts,
                 desc="Experiment 3: Sliding window test inference"):
    prob = predict_note_probability_sliding_window(
        text,
        batch_size=16
    )
    test_probs_exp3.append(prob)

test_probs_exp3 = np.array(test_probs_exp3)

print("Test inference complete.")
print("Number of test predictions:", len(test_probs_exp3))

Experiment 3: Sliding window test inference: 100%|██████████| 5460/5460 [24:28<00:00,  3.72it/s]

Test inference complete.
Number of test predictions: 5460


- Apply Best Validation Threshold

In [39]:
best_exp3_threshold = 0.85

test_preds_exp3 = (
    test_probs_exp3 >= best_exp3_threshold
).astype(int)

- Final Test Evaluation

In [40]:
exp3_accuracy = accuracy_score(test_labels, test_preds_exp3)
exp3_precision = precision_score(test_labels, test_preds_exp3)
exp3_recall = recall_score(test_labels, test_preds_exp3)
exp3_f1 = f1_score(test_labels, test_preds_exp3)
exp3_auc = roc_auc_score(test_labels, test_probs_exp3)

print("=" * 60)
print("Experiment 3 Final Test Results")
print("=" * 60)

print(f"Accuracy : {exp3_accuracy:.4f}")
print(f"Precision: {exp3_precision:.4f}")
print(f"Recall   : {exp3_recall:.4f}")
print(f"F1-score : {exp3_f1:.4f}")
print(f"ROC-AUC  : {exp3_auc:.4f}")

Experiment 3 Final Test Results
Accuracy : 0.8154
Precision: 0.3382
Recall   : 0.6875
F1-score : 0.4534
ROC-AUC  : 0.8217


In [41]:

# Confusion Matrix

cm = confusion_matrix(test_labels, test_preds_exp3)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"]
)

cm_df

,Predicted Negative,Predicted Positive
Actual Negative,4034,818
Actual Positive,190,418


In [42]:

# Classification Report

print(
    classification_report(
        test_labels,
        test_preds_exp3,
        digits=4
    )
)

              precision    recall  f1-score   support

           0     0.9550    0.8314    0.8889      4852
           1     0.3382    0.6875    0.4534       608

    accuracy                         0.8154      5460
   macro avg     0.6466    0.7595    0.6712      5460
weighted avg     0.8863    0.8154    0.8404      5460




- Save Experiment 3 Results


In [43]:
exp3_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"
    ],
    "Value": [
        exp3_accuracy,
        exp3_precision,
        exp3_recall,
        exp3_f1,
        exp3_auc
    ]
})

exp3_results.to_csv(
    DATA_PATH + "experiment3_test_results.csv",
    index=False
)

predictions_df = pd.DataFrame({
    "True_Label": test_labels,
    "Predicted_Probability": test_probs_exp3,
    "Predicted_Label": test_preds_exp3
})

predictions_df.to_csv(
    DATA_PATH + "experiment3_test_predictions.csv",
    index=False
)

print("Experiment 3 test results saved successfully.")

Experiment 3 test results saved successfully.


In [45]:

exp3_threshold_df.to_csv(
    RESULTS_PATH + "experiment3_validation_thresholds.csv",
    index=False
)

exp3_results.to_csv(
    RESULTS_PATH + "experiment3_test_results.csv",
    index=False
)

pd.DataFrame({
    "True_Label": test_labels,
    "Predicted_Probability": test_probs_exp3,
    "Predicted_Label": test_preds_exp3
}).to_csv(
    RESULTS_PATH + "experiment3_test_predictions.csv",
    index=False
)

print("Experiment 3 outputs copied to shared evaluation folder.")

Experiment 3 outputs copied to shared evaluation folder.


## Experiment 3 Summary

In this experiment, the ClinicalBERT model was evaluated using a sliding window inference strategy to overcome the 512-token input limitation. Each clinical note was divided into overlapping token chunks, and the maximum predicted probability across all chunks was used as the final document-level prediction.

The sliding window approach achieved a test accuracy of **81.54%**, with a **precision of 33.82%**, **recall of 68.75%**, **F1-score of 45.34%**, and **ROC-AUC of 0.8217**. Compared to the baseline ClinicalBERT model, this approach improved the model's ability to identify positive sepsis cases (higher recall), but also increased the number of false-positive predictions, resulting in lower precision and F1-score.

Overall, while incorporating the full clinical note provided additional contextual information, the sliding window strategy did not outperform the baseline ClinicalBERT model in terms of overall classification performance. These findings suggest that using the first 512 tokens was sufficient to capture the most informative features for this sepsis prediction task.

## Key Findings

| Experiment       | Main Idea                          | Key Outcome                                                                                                                                 |
| ---------------- | ---------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------- |
| **Experiment 1** | ClinicalBERT baseline (512 tokens) | Best overall performance and strongest balance between precision and recall.                                                                |
| **Experiment 2** | Threshold optimization             | Demonstrated how decision thresholds affect the precision–recall trade-off and identified the optimal operating threshold.                  |
| **Experiment 3** | Sliding window inference           | Improved recall by processing complete clinical notes but increased false positives, resulting in lower overall F1-score than the baseline. |


# Notebook Summary

This notebook implemented and evaluated transformer-based models for early sepsis prediction using ClinicalBERT. Three complementary experiments were conducted to investigate different inference strategies and decision thresholds.

- **Experiment 1** established the baseline ClinicalBERT model by fine-tuning the model using the first 512 tokens of each clinical note.
- **Experiment 2** explored threshold optimization to identify the probability threshold that provided the best balance between precision and recall without modifying the trained model.
- **Experiment 3** introduced a sliding window inference strategy, allowing the model to process complete clinical notes by aggregating predictions from overlapping text segments.

The experimental results demonstrated that the baseline ClinicalBERT model achieved the strongest overall performance, while threshold optimization provided insight into the trade-offs between sensitivity and precision. Although the sliding window approach successfully incorporated information from the entire clinical note and improved recall, it also increased false-positive predictions and did not improve the overall F1-score or classification performance.

The outputs generated in this notebook, including trained models, prediction probabilities, evaluation metrics, and experiment-specific results, will be used in the next notebook to perform a comprehensive comparison of all developed models and identify the best-performing approach for early sepsis prediction.